In [16]:
import pandas as pd

emissions_df = pd.read_csv("./data/co2_pcap_cons.csv")
emissions_df

,country,1800,1801,1802,1803,1804,1805,1806,1807,1808,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,Afghanistan,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.28,0.253,0.262,0.245,0.247,0.254,0.261,0.261,0.279,0.284
1,Angola,0.009,0.009,0.009,0.009,0.009,0.009,0.010,0.010,0.010,...,1.28,1.640,1.220,1.180,1.150,1.120,1.150,1.120,1.200,1.230
2,Albania,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,2.27,2.250,2.040,2.010,2.130,2.080,2.050,2.000,2.120,2.100
3,Andorra,0.333,0.335,0.337,0.340,0.342,0.345,0.347,0.350,0.352,...,5.9,5.830,5.970,6.070,6.270,6.120,6.060,5.630,5.970,5.910
4,UAE,0.063,0.063,0.064,0.064,0.064,0.064,0.065,0.065,0.065,...,27,26.800,27.000,26.700,23.900,23.500,21.200,19.700,20.700,21.100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,Samoa,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,1.04,1.090,1.210,1.260,1.290,1.320,1.370,1.310,1.400,1.430
190,Yemen,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,0.994,0.937,0.480,0.377,0.363,0.356,0.365,0.362,0.387,0.395
191,South Africa,0.003,0.003,0.004,0.004,0.004,0.004,0.004,0.004,0.004,...,6.2,6.100,5.760,5.680,5.550,5.420,5.680,5.110,5.080,5.180
192,Zambia,0.255,0.256,0.257,0.258,0.259,0.260,0.261,0.262,0.263,...,0.511,0.560,0.519,0.471,0.474,0.467,0.487,0.388,0.416,0.424


In [17]:
!curl -s https://wagon-public-datasets.s3.amazonaws.com/02-Data-Toolkit/02-Data-Sourcing/iso2_with_region.csv > data/iso2_with_region.csv
iso_df = pd.read_csv('data/iso2_with_region.csv', na_filter=False)
iso_df.head()

,Name,Code,Full Name,Region
0,Afghanistan,AF,Afghanistan,Asia
1,Åland Islands,AX,Åland Islands,Europe
2,Albania,AL,Albania,Europe
3,Algeria,DZ,Algeria,Africa
4,American Samoa,AS,American Samoa,Oceania


In [18]:
emissions_df = emissions_df.merge(iso_df, left_on="country", right_on="Name")
emissions_df = emissions_df.drop(columns=["country"])
emissions_df

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2017,2018,2019,2020,2021,2022,Name,Code,Full Name,Region
0,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.247,0.254,0.261,0.261,0.279,0.284,Afghanistan,AF,Afghanistan,Asia
1,0.009,0.009,0.009,0.009,0.009,0.009,0.010,0.010,0.010,0.010,...,1.150,1.120,1.150,1.120,1.200,1.230,Angola,AO,Angola,Africa
2,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,2.130,2.080,2.050,2.000,2.120,2.100,Albania,AL,Albania,Europe
3,0.333,0.335,0.337,0.340,0.342,0.345,0.347,0.350,0.352,0.355,...,6.270,6.120,6.060,5.630,5.970,5.910,Andorra,AD,Andorra,Europe
4,0.063,0.063,0.064,0.064,0.064,0.064,0.065,0.065,0.065,0.065,...,23.900,23.500,21.200,19.700,20.700,21.100,UAE,AE,UAE,Asia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,1.290,1.320,1.370,1.310,1.400,1.430,Samoa,WS,Samoa,Oceania
190,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,0.363,0.356,0.365,0.362,0.387,0.395,Yemen,YE,Yemen,Asia
191,0.003,0.003,0.004,0.004,0.004,0.004,0.004,0.004,0.004,0.004,...,5.550,5.420,5.680,5.110,5.080,5.180,South Africa,ZA,South Africa,Africa
192,0.255,0.256,0.257,0.258,0.259,0.260,0.261,0.262,0.263,0.264,...,0.474,0.467,0.487,0.388,0.416,0.424,Zambia,ZM,Zambia,Africa


In [19]:
# --- BACKUP: fill the Population column for ALL countries ---
import requests
import pandas as pd

codes, pops = [], []
page = 1
while True:
    response = requests.get(
        "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL",
        params={"format": "json", "per_page": 500, "page": page},
        timeout=30,
    ).json()
    codes.extend(r["country"]["id"] for r in response[1])
    pops.extend(r["value"] for r in response[1])
    if page >= response[0]["pages"]:
        break
    page += 1

# Map Code -> Population onto the existing column (keeps emissions_df's structure intact)
emissions_df["Population"] = emissions_df["Code"].map(dict(zip(codes, pops)))

print(f"Fetched {len(codes)} World Bank rows over {page} page(s)")
print(f"Population filled for {emissions_df['Population'].notna().sum()} / {len(emissions_df)} countries")

Fetched 17490 World Bank rows over 35 page(s)
Population filled for 192 / 194 countries


In [20]:
emissions_df

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2018,2019,2020,2021,2022,Name,Code,Full Name,Region,Population
0,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.254,0.261,0.261,0.279,0.284,Afghanistan,AF,Afghanistan,Asia,9035043.0
1,0.009,0.009,0.009,0.009,0.009,0.009,0.010,0.010,0.010,0.010,...,1.120,1.150,1.120,1.200,1.230,Angola,AO,Angola,Africa,5231654.0
2,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,2.080,2.050,2.000,2.120,2.100,Albania,AL,Albania,Europe,1608800.0
3,0.333,0.335,0.337,0.340,0.342,0.345,0.347,0.350,0.352,0.355,...,6.120,6.060,5.630,5.970,5.910,Andorra,AD,Andorra,Europe,9510.0
4,0.063,0.063,0.064,0.064,0.064,0.064,0.065,0.065,0.065,0.065,...,23.500,21.200,19.700,20.700,21.100,UAE,AE,UAE,Asia,131334.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,1.320,1.370,1.310,1.400,1.430,Samoa,WS,Samoa,Oceania,112490.0
190,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002,...,0.356,0.365,0.362,0.387,0.395,Yemen,YE,Yemen,Asia,5532301.0
191,0.003,0.003,0.004,0.004,0.004,0.004,0.004,0.004,0.004,0.004,...,5.420,5.680,5.110,5.080,5.180,South Africa,ZA,South Africa,Africa,16440172.0
192,0.255,0.256,0.257,0.258,0.259,0.260,0.261,0.262,0.263,0.264,...,0.467,0.487,0.388,0.416,0.424,Zambia,ZM,Zambia,Africa,3153729.0


In [21]:
# Make a list of column names, starting with the country identification
new_order = ["Code", "Name", "Full Name", "Region", "Population"]
# Add the columns for the years
new_order += list(emissions_df.columns)[:-5]
# Use the list to reorder the dataframe
emissions_df = emissions_df[new_order]
emissions_df.head(2)

,Code,Name,Full Name,Region,Population,1800,1801,1802,1803,1804,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,AF,Afghanistan,Afghanistan,Asia,9035043.0,0.001,0.001,0.001,0.001,0.001,...,0.28,0.253,0.262,0.245,0.247,0.254,0.261,0.261,0.279,0.284
1,AO,Angola,Angola,Africa,5231654.0,0.009,0.009,0.009,0.009,0.009,...,1.28,1.640,1.220,1.180,1.150,1.120,1.150,1.120,1.200,1.230


In [29]:
# Keep relevant columns only, incl. emissions for 2022 only
emissions_2022 = emissions_df[["Full Name", "Region", "2022", "Population"]].copy()
# Calculate total emissions = emissions per capita x population
emissions_2022["Total Emissions"] = (
    emissions_2022["2022"] * emissions_2022["Population"]
)

In [30]:
emissions_2022

,Full Name,Region,2022,Population,Total Emissions
0,Afghanistan,Asia,0.284,9035043.0,2.565952e+06
1,Angola,Africa,1.230,5231654.0,6.434934e+06
2,Albania,Europe,2.100,1608800.0,3.378480e+06
3,Andorra,Europe,5.910,9510.0,5.620410e+04
4,UAE,Asia,21.100,131334.0,2.771147e+06
...,...,...,...,...,...
189,Samoa,Oceania,1.430,112490.0,1.608607e+05
190,Yemen,Asia,0.395,5532301.0,2.185259e+06
191,South Africa,Africa,5.180,16440172.0,8.516009e+07
192,Zambia,Africa,0.424,3153729.0,1.337181e+06
